# Greedy Hill Climbing for the N-Queens Problem

## Author Details
- **Author:** Ramya Mercy Rajan  
- **Course:** MSc Software Engineering  
- **University:** University of Europe for Applied Sciences  
- **Professor:** Raja Hashim Ali  

---

# Algorithm Overview

Greedy Hill Climbing is a local search algorithm that tries to
reduce queen conflicts step by step.

The algorithm starts with a random arrangement of queens on the board,
with one queen placed in each row.

The following steps are repeated:

1. Check each queen on the board.
2. Find the best column in the same row that gives the least conflicts.
3. Move the queen if the new position improves the board.
4. Continue this process until:
   - all conflicts become zero, or
   - no better move can be found.

If the algorithm gets stuck and cannot improve the solution further,
it reaches a local minimum. In that case, the board is restarted
with a new random arrangement.

---

# Optimization Used

A simple implementation checks all queens every time conflicts
need to be calculated, which becomes slow for large board sizes.

To improve performance, three arrays were used to store the number
of queens present in each column and diagonal.

- col[c] stores queens in column c
- diag1[r-c+N-1] stores queens in the / diagonal
- diag2[r+c] stores queens in the \ diagonal

Using these arrays, conflicts can be checked much faster without
scanning the whole board repeatedly.

This reduced the runtime significantly for larger values of \(N\).

---

# Time Complexity

The overall time complexity of the algorithm is:

\[
O(\text{restarts} \times \text{passes} \times N^2)
\]

because every queen may check all columns during each pass.

In [8]:
import time
import random
import psutil
import os

In [9]:
def measure_memory():
    process = psutil.Process(os.getpid())
    return process.memory_info().rss / (1024 * 1024)


In [10]:


def build_counts(board):
    N     = len(board)
    col   = [0] * N
    diag1 = [0] * (2 * N)  
    diag2 = [0] * (2 * N)  
    for r, c in enumerate(board):
        col[c]             += 1
        diag1[r - c + N-1] += 1
        diag2[r + c]       += 1
    return col, diag1, diag2

In [11]:
def row_conflicts(r, c, col, diag1, diag2, N):
    return (col[c] - 1) + (diag1[r - c + N-1] - 1) + (diag2[r + c] - 1)

In [12]:
def total_conflicts(board, col, diag1, diag2):
    N = len(board)
    return sum(row_conflicts(r, board[r], col, diag1, diag2, N)
               for r in range(N)) // 2

In [13]:
## Greedy Hill Climbing
def greedy_hill_climbing(N, max_restarts=200):
    start      = time.time()
    mem_before = measure_memory()

    for restart in range(max_restarts):

        
        board = [random.randint(0, N - 1) for _ in range(N)]
        col, diag1, diag2 = build_counts(board)

        while True:
            
            if total_conflicts(board, col, diag1, diag2) == 0:
                return {
                    "solution":  board,
                    "time_sec":  round(time.time() - start, 4),
                    "memory_MB": round(measure_memory() - mem_before, 4),
                    "restarts":  restart,
                }

            improved = False

           
            for r in range(N):
                old_c     = board[r]
                best_c    = old_c
                best_conf = row_conflicts(r, old_c, col, diag1, diag2, N)

                
                col[old_c]             -= 1
                diag1[r - old_c + N-1] -= 1
                diag2[r + old_c]       -= 1

                
                for c in range(N):
                    col[c]             += 1
                    diag1[r - c + N-1] += 1
                    diag2[r + c]       += 1

                    conf = (col[c] - 1) + (diag1[r - c + N-1] - 1) \
                                        + (diag2[r + c] - 1)
                    if conf < best_conf:
                        best_conf = conf
                        best_c    = c
                        improved  = True

                    col[c]             -= 1
                    diag1[r - c + N-1] -= 1
                    diag2[r + c]       -= 1

              
                board[r] = best_c
                col[best_c]             += 1
                diag1[r - best_c + N-1] += 1
                diag2[r + best_c]       += 1

         
            if not improved:
                break

    return {
        "solution":  None,
        "time_sec":  round(time.time() - start, 4),
        "memory_MB": round(measure_memory() - mem_before, 4),
        "restarts":  max_restarts,
    }

In [14]:
def print_board(board):
    N = len(board)
    print()
    for row in range(N):
        line = ""
        for col in range(N):
            line += " Q " if board[row] == col else " . "
        print(line)
    print()

In [ ]:
def run_experiments():
    test_sizes = [10, 30, 50, 100, 200, 500]

    print("=" * 60)
    print("  N-Queens — Greedy Hill Climbing")
    print("=" * 60)

    for N in test_sizes:
        print(f"\n>>> N = {N}")
        result = greedy_hill_climbing(N, max_restarts=200)

        solved = result["solution"] is not None
        print(f"  Solved          : {solved}")
        print(f"  Restarts used   : {result['restarts']}")
        print(f"  Time            : {result['time_sec']} s")
        print(f"  Memory delta    : {result['memory_MB']} MB")

        if solved:
            print(f"  Solution        : {result['solution']}")
            if N <= 10:
                print_board(result["solution"])
        else:
            print(f"  Result          : No solution found within "
                  f"{result['restarts']} restarts.")


if __name__ == "__main__":
    run_experiments()

  N-Queens — Greedy Hill Climbing

>>> N = 10
  Solved          : True
  Restarts used   : 10
  Time            : 0.0245 s
  Memory delta    : 0.1875 MB
  Solution        : [5, 7, 1, 6, 8, 2, 0, 3, 9, 4]

 .  .  .  .  .  Q  .  .  .  . 
 .  .  .  .  .  .  .  Q  .  . 
 .  Q  .  .  .  .  .  .  .  . 
 .  .  .  .  .  .  Q  .  .  . 
 .  .  .  .  .  .  .  .  Q  . 
 .  .  Q  .  .  .  .  .  .  . 
 Q  .  .  .  .  .  .  .  .  . 
 .  .  .  Q  .  .  .  .  .  . 
 .  .  .  .  .  .  .  .  .  Q 
 .  .  .  .  Q  .  .  .  .  . 


>>> N = 30
  Solved          : True
  Restarts used   : 50
  Time            : 0.157 s
  Memory delta    : 0.0156 MB
  Solution        : [3, 14, 16, 11, 13, 6, 29, 24, 2, 28, 20, 8, 10, 0, 19, 1, 5, 12, 9, 26, 22, 27, 21, 23, 17, 15, 18, 7, 4, 25]

>>> N = 50
  Solved          : True
  Restarts used   : 59
  Time            : 0.4481 s
  Memory delta    : 0.0547 MB
  Solution        : [4, 19, 11, 18, 37, 24, 9, 20, 1, 29, 42, 32, 14, 48, 40, 10, 0, 31, 28, 13, 43, 26, 38, 45, 12,